In [ ]:
"""
cnn_model.py
============
Train a quantized CNN on MNIST, export weights as numpy arrays,
save the full Keras model, and run hls4ml comparison.

CNN architecture (matches cnn_top.vhd exactly):
    Input:  28x28x1  (8-bit)
    Conv1:  8 filters, 3x3, valid  -> 26x26x8
    ReLU + MaxPool 2x2             -> 13x13x8
    Conv2:  16 filters, 3x3, valid -> 11x11x16
    ReLU + MaxPool 2x2             -> 5x5x16
    Flatten                        -> 400
    FC1:    400 -> 64
    ReLU
    FC2:    64 -> 10
    Softmax

Exported files:
    weights/conv1_weights.npy   shape: (8,  1,  3, 3)   int8
    weights/conv1_bias.npy      shape: (8,)              int32
    weights/conv2_weights.npy   shape: (16, 8,  3, 3)   int8
    weights/conv2_bias.npy      shape: (16,)             int32
    weights/fc1_weights.npy     shape: (64, 400)         int8
    weights/fc1_bias.npy        shape: (64,)             int32
    weights/fc2_weights.npy     shape: (10, 64)          int8
    weights/fc2_bias.npy        shape: (10,)             int32
    weights/scales.npy          dict of scale factors    float32
    cnn_mnist.keras             full Keras model for hls4ml
"""

import os
import sys
import shutil
import numpy as np
import tensorflow as tf
from tensorflow import keras

# ----------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------
EPOCHS      = 10
BATCH_SIZE  = 128
WEIGHT_BITS = 8
W_MAX       = 2 ** (WEIGHT_BITS - 1) - 1   # 127
W_MIN       = -(2 ** (WEIGHT_BITS - 1))     # -128
WEIGHTS_DIR = os.path.join(os.getcwd(), "weights")

os.makedirs(WEIGHTS_DIR, exist_ok=True)


# ----------------------------------------------------------------
# Quantize weights to 8-bit signed
# ----------------------------------------------------------------
def quantize_weights(w_float, name=""):
    max_abs = np.max(np.abs(w_float))
    if max_abs == 0:
        return np.zeros_like(w_float, dtype=np.int8), 1.0
    scale = W_MAX / max_abs
    w_int = np.clip(np.round(w_float * scale), W_MIN, W_MAX).astype(np.int8)
    print(f"  {name}: max_abs={max_abs:.4f}  scale={scale:.4f}  "
          f"range=[{w_int.min()}, {w_int.max()}]")
    return w_int, scale


def quantize_bias(b_float, scale, name=""):
    b_int = np.clip(np.round(b_float * scale),
                    -(2**31), 2**31 - 1).astype(np.int32)
    print(f"  {name}: range=[{b_int.min()}, {b_int.max()}]")
    return b_int


# ----------------------------------------------------------------
# Build Keras model
# ----------------------------------------------------------------
def build_model():
    model = keras.Sequential([
        keras.layers.Input(shape=(28, 28, 1)),
        keras.layers.Conv2D(8,  (3, 3), padding='valid', activation='relu', name='conv1'),
        keras.layers.MaxPooling2D((2, 2), name='pool1'),
        keras.layers.Conv2D(16, (3, 3), padding='valid', activation='relu', name='conv2'),
        keras.layers.MaxPooling2D((2, 2), name='pool2'),
        keras.layers.Flatten(name='flatten'),
        keras.layers.Dense(64, activation='relu',    name='fc1'),
        keras.layers.Dense(10, activation='softmax', name='fc2'),
    ], name='cnn_mnist')
    return model


# ----------------------------------------------------------------
# Train model
# ----------------------------------------------------------------
def train_model():
    print("=" * 60)
    print("Loading MNIST dataset")
    print("=" * 60)

    (x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
    x_train = x_train.astype(np.float32) / 255.0
    x_test  = x_test.astype(np.float32)  / 255.0
    x_train = x_train[..., np.newaxis]
    x_test  = x_test[..., np.newaxis]
    print(f"  Train: {x_train.shape}  Test: {x_test.shape}")

    print()
    print("=" * 60)
    print("Building model")
    print("=" * 60)
    model = build_model()
    model.summary()

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    print()
    print("=" * 60)
    print(f"Training for {EPOCHS} epochs")
    print("=" * 60)
    model.fit(
        x_train, y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_split=0.1,
        verbose=1
    )

    print()
    print("=" * 60)
    print("Evaluating on test set")
    print("=" * 60)
    test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
    print(f"  Test accuracy: {test_acc*100:.2f}%")
    print(f"  Test loss:     {test_loss:.4f}")
    if test_acc < 0.97:
        print("  WARNING: accuracy below 97% — consider retraining")
    else:
        print("  OK: accuracy >= 97%")

    return model, x_test, y_test, test_acc


# ----------------------------------------------------------------
# Export quantized weights
# ----------------------------------------------------------------
def export_weights(model):
    print()
    print("=" * 60)
    print("Quantizing and exporting weights")
    print("=" * 60)

    scales = {}

    # Conv1
    conv1_layer   = model.get_layer('conv1')
    conv1_w_keras = conv1_layer.get_weights()[0]  # (3,3,1,8)
    conv1_b_keras = conv1_layer.get_weights()[1]
    conv1_w = np.transpose(conv1_w_keras, (3, 2, 0, 1))  # (8,1,3,3)
    print("Conv1 weights:")
    conv1_w_int, conv1_scale = quantize_weights(conv1_w, "conv1_w")
    print("Conv1 bias:")
    conv1_b_int = quantize_bias(conv1_b_keras, conv1_scale, "conv1_b")
    scales['conv1'] = conv1_scale
    np.save(os.path.join(WEIGHTS_DIR, "conv1_weights.npy"), conv1_w_int)
    np.save(os.path.join(WEIGHTS_DIR, "conv1_bias.npy"),    conv1_b_int)
    print(f"  Saved conv1_weights.npy  shape={conv1_w_int.shape}")

    # Conv2
    conv2_layer   = model.get_layer('conv2')
    conv2_w_keras = conv2_layer.get_weights()[0]  # (3,3,8,16)
    conv2_b_keras = conv2_layer.get_weights()[1]
    conv2_w = np.transpose(conv2_w_keras, (3, 2, 0, 1))  # (16,8,3,3)
    print("Conv2 weights:")
    conv2_w_int, conv2_scale = quantize_weights(conv2_w, "conv2_w")
    print("Conv2 bias:")
    conv2_b_int = quantize_bias(conv2_b_keras, conv2_scale, "conv2_b")
    scales['conv2'] = conv2_scale
    np.save(os.path.join(WEIGHTS_DIR, "conv2_weights.npy"), conv2_w_int)
    np.save(os.path.join(WEIGHTS_DIR, "conv2_bias.npy"),    conv2_b_int)
    print(f"  Saved conv2_weights.npy  shape={conv2_w_int.shape}")

    # FC1
    fc1_layer   = model.get_layer('fc1')
    fc1_w_keras = fc1_layer.get_weights()[0]  # (400,64)
    fc1_b_keras = fc1_layer.get_weights()[1]
    fc1_w = np.transpose(fc1_w_keras, (1, 0))  # (64,400)
    print("FC1 weights:")
    fc1_w_int, fc1_scale = quantize_weights(fc1_w, "fc1_w")
    print("FC1 bias:")
    fc1_b_int = quantize_bias(fc1_b_keras, fc1_scale, "fc1_b")
    scales['fc1'] = fc1_scale
    np.save(os.path.join(WEIGHTS_DIR, "fc1_weights.npy"), fc1_w_int)
    np.save(os.path.join(WEIGHTS_DIR, "fc1_bias.npy"),    fc1_b_int)
    print(f"  Saved fc1_weights.npy    shape={fc1_w_int.shape}")

    # FC2
    fc2_layer   = model.get_layer('fc2')
    fc2_w_keras = fc2_layer.get_weights()[0]  # (64,10)
    fc2_b_keras = fc2_layer.get_weights()[1]
    fc2_w = np.transpose(fc2_w_keras, (1, 0))  # (10,64)
    print("FC2 weights:")
    fc2_w_int, fc2_scale = quantize_weights(fc2_w, "fc2_w")
    print("FC2 bias:")
    fc2_b_int = quantize_bias(fc2_b_keras, fc2_scale, "fc2_b")
    scales['fc2'] = fc2_scale
    np.save(os.path.join(WEIGHTS_DIR, "fc2_weights.npy"), fc2_w_int)
    np.save(os.path.join(WEIGHTS_DIR, "fc2_bias.npy"),    fc2_b_int)
    print(f"  Saved fc2_weights.npy    shape={fc2_w_int.shape}")

    np.save(os.path.join(WEIGHTS_DIR, "scales.npy"), scales)
    print(f"  Saved scales.npy         {scales}")

    return scales


# ----------------------------------------------------------------
# Verify exported weights
# ----------------------------------------------------------------
def verify_quantized(model, x_test, y_test, scales):
    print()
    print("=" * 60)
    print("Verifying quantized weights (100 samples)")
    print("=" * 60)

    conv1_w = np.load(os.path.join(WEIGHTS_DIR, "conv1_weights.npy"))
    conv2_w = np.load(os.path.join(WEIGHTS_DIR, "conv2_weights.npy"))
    fc1_w   = np.load(os.path.join(WEIGHTS_DIR, "fc1_weights.npy"))
    fc2_w   = np.load(os.path.join(WEIGHTS_DIR, "fc2_weights.npy"))

    print(f"  conv1: shape={conv1_w.shape}  dtype={conv1_w.dtype}")
    print(f"  conv2: shape={conv2_w.shape}  dtype={conv2_w.dtype}")
    print(f"  fc1:   shape={fc1_w.shape}  dtype={fc1_w.dtype}")
    print(f"  fc2:   shape={fc2_w.shape}  dtype={fc2_w.dtype}")

    preds = model.predict(x_test[:100], verbose=0)
    float_acc = np.mean(np.argmax(preds, axis=1) == y_test[:100])
    print(f"  Float model accuracy on 100 samples: {float_acc*100:.1f}%")


# ----------------------------------------------------------------
# Patch hls4ml for Keras 3 compatibility
# ----------------------------------------------------------------
def patch_hls4ml():
    print()
    print("=" * 60)
    print("Patching hls4ml for Keras 3 compatibility")
    print("=" * 60)

    # Patch keras_to_hls.py
    f1 = "/home/mas47153/.local/lib/python3.12/site-packages/hls4ml/converters/keras_to_hls.py"
    src = open(f1).read()
    if "or 'batch_shape' in keras_layer['config']:" not in src:
        src = src.replace(
            "if 'batch_input_shape' in keras_layer['config']:",
            "if 'batch_input_shape' in keras_layer['config'] or 'batch_shape' in keras_layer['config']:"
        )
        src = src.replace(
            "input_shapes = [keras_layer['config']['batch_input_shape']]",
            "input_shapes = [keras_layer['config'].get('batch_input_shape', keras_layer['config'].get('batch_shape'))]"
        )
        open(f1, 'w').write(src)
        print("  Patched keras_to_hls.py")
    else:
        print("  keras_to_hls.py already patched")

    # Patch core.py
    f2 = "/home/mas47153/.local/lib/python3.12/site-packages/hls4ml/converters/keras/core.py"
    src = open(f2).read()

    src = src.replace(
        "layer['input_shape'] = keras_layer['config']['batch_input_shape'][1:]",
        "layer['input_shape'] = keras_layer['config'].get('batch_input_shape', keras_layer['config'].get('batch_shape'))[1:]"
    )
    src = src.replace(
        "output_shape = keras_layer['config']['batch_input_shape']",
        "output_shape = keras_layer['config'].get('batch_input_shape', keras_layer['config'].get('batch_shape'))"
    )
    src = src.replace(
        "dtype = keras_layer['config']['dtype']",
        "dtype = keras_layer['config'].get('dtype', 'float32')\n    if isinstance(dtype, dict):\n        dtype = dtype.get('config', {}).get('name', 'float32')"
    )
    open(f2, 'w').write(src)
    print("  Patched keras/core.py")

    # Force reload all hls4ml modules
    to_remove = [k for k in sys.modules if 'hls4ml' in k]
    for k in to_remove:
        del sys.modules[k]
    print(f"  Reloaded {len(to_remove)} hls4ml modules from disk")


# ----------------------------------------------------------------
# hls4ml conversion and comparison
# ----------------------------------------------------------------
def run_hls4ml_comparison(model, x_test, y_test, test_acc):
    print()
    print("=" * 60)
    print("hls4ml Conversion and Comparison")
    print("=" * 60)

    import hls4ml

    HLS_PROJECT_DIR = os.path.join(os.getcwd(), "hls4ml_cnn_project")
    FPGA_PART       = "xc7z020clg484-1"
    CLOCK_PERIOD_NS = 10
    REUSE_FACTOR    = 1

    # Build config manually
    config = {
        'Model': {'Precision': 'ap_fixed<16,8>', 'ReuseFactor': REUSE_FACTOR},
        'LayerName': {}
    }
    for lyr in model.layers:
        n, lt = lyr.name, lyr.__class__.__name__
        if lt in ('Conv2D', 'Dense'):
            config['LayerName'][n] = {
                'Precision': {
                    'weight': 'ap_fixed<16,8>',
                    'bias'  : 'ap_fixed<16,8>',
                    'result': 'ap_fixed<16,8>',
                    'accum' : 'ap_fixed<32,16>',
                },
                'ReuseFactor': REUSE_FACTOR
            }
        elif lt == 'MaxPooling2D':
            config['LayerName'][n] = {'Precision': {'result': 'ap_fixed<16,8>'}}
        elif lt in ('Flatten', 'InputLayer'):
            config['LayerName'][n] = {'Precision': {'result': 'ap_ufixed<16,10>'}}

    print("  Layer config:")
    for n, c in config['LayerName'].items():
        p = c.get('Precision', {})
        print(f"    {n:<12}  w={p.get('weight','def'):16s}  acc={p.get('accum','def')}")

    # Convert
    print("\n  Converting to HLS...")
    if os.path.exists(HLS_PROJECT_DIR):
        shutil.rmtree(HLS_PROJECT_DIR)

    hls_model = hls4ml.converters.convert_from_keras_model(
        model,
        hls_config   = config,
        output_dir   = HLS_PROJECT_DIR,
        backend      = 'Vivado',
        part         = FPGA_PART,
        clock_period = CLOCK_PERIOD_NS,
        io_type      = 'io_stream'
    )
    hls_model.compile()
    print("  Compiled OK")

    # Validate
    print("\n  Validating (1000 samples)...")
    kp = np.argmax(model.predict(x_test[:1000], verbose=0), axis=1)
    hp = np.argmax(hls_model.predict(x_test[:1000]), axis=1)
    ka = np.mean(kp == y_test[:1000])
    ha = np.mean(hp == y_test[:1000])
    mr = np.mean(kp == hp)
    print(f"  Keras  accuracy : {ka*100:.2f}%")
    print(f"  hls4ml accuracy : {ha*100:.2f}%")
    print(f"  Prediction match: {mr*100:.2f}%")
    print("  OK" if mr >= 0.95 else "  WARNING: match below 95%")

    # Synthesis
    print("\n  Running HLS synthesis (takes several minutes)...")
    os.environ['PATH'] = '/opt/Xilinx/Vivado/2020.2/bin:' + os.environ['PATH']
    tcl = os.path.join(HLS_PROJECT_DIR, 'build_prj.tcl')
    if os.path.exists(tcl):
        lines = [l for l in open(tcl).read().splitlines()
                 if 'config_array_partition' not in l]
        open(tcl, 'w').write('\n'.join(lines) + '\n')
        print("  Patched build_prj.tcl for Vivado 2020.2")

    synth_ok = False
    hls_report = {}
    try:
        hls_model.build(csim=False, synth=True, vsynth=False)
        print("  Synthesis complete")
        synth_ok = True
    except Exception as e:
        print(f"  Synthesis error: {e}")
        print("  Source Vivado first:")
        print("  source /opt/Xilinx/Vivado/2020.2/settings64.sh")

    if synth_ok:
        try:
            hls_report = hls4ml.report.read_vivado_report(HLS_PROJECT_DIR)
        except Exception as e:
            print(f"  Could not parse report: {e}")

    # Comparison table
    print()
    print("=" * 60)
    print("Resource Comparison: Custom VHDL vs hls4ml")
    print("=" * 60)

    vhdl = dict(DSP=199, RAMB36=2, RAMB18=1, LUT=42474, FF=6574, Fmax=102.6)

    def fh(key):
        return str(hls_report.get(key, '(see rpt)'))

    print(f"\n{'Resource':<16}{'Custom VHDL':>14}{'hls4ml':>14}{'Device Max':>14}")
    print("-" * 60)
    print(f"{'DSP48E1':<16}{vhdl['DSP']:>14}  {fh('DSP48E'):>13}  {'220':>13}")
    print(f"{'RAMB36':<16}{vhdl['RAMB36']:>14}  {fh('BRAM_36K'):>13}  {'140':>13}")
    print(f"{'RAMB18':<16}{vhdl['RAMB18']:>14}  {fh('BRAM_18K'):>13}  {'280':>13}")
    print(f"{'LUT':<16}{vhdl['LUT']:>14}  {fh('LUT'):>13}  {'53200':>13}")
    print(f"{'FF':<16}{vhdl['FF']:>14}  {fh('FF'):>13}  {'106400':>13}")
    print(f"{'Fmax MHz':<16}{vhdl['Fmax']:>14.1f}  {'(see rpt)':>13}  {'100 target':>13}")
    print(f"{'Accuracy %':<16}{test_acc*100:>14.2f}  {ha*100:>13.2f}")

    rpt_path = os.path.join(HLS_PROJECT_DIR, 'cnn_mnist_prj', 'solution1',
                            'syn', 'report', 'cnn_mnist_csynth.rpt')
    if os.path.exists(rpt_path):
        print("\n" + "=" * 60)
        print("Full HLS Synthesis Report:")
        print("=" * 60)
        print(open(rpt_path).read())


# ----------------------------------------------------------------
# Main
# ----------------------------------------------------------------
if __name__ == "__main__":
    print()
    print("CNN Model Training and Weight Export")
    print("=====================================")
    print(f"TensorFlow version: {tf.__version__}")
    print(f"Weights directory : {WEIGHTS_DIR}")
    print()

    # Train
    model, x_test, y_test, test_acc = train_model()

    # Export numpy weights (for VHDL testbench)
    scales = export_weights(model)

    # Verify
    verify_quantized(model, x_test, y_test, scales)

    # Save full Keras model (for hls4ml)
    print()
    print("=" * 60)
    print("Saving full Keras model")
    print("=" * 60)
    model.save("cnn_mnist.keras")
    print("  Saved cnn_mnist.keras")

    print()
    print("=" * 60)
    print("DONE — Weight export complete")
    print("=" * 60)
    print("Exported files:")
    for f in sorted(os.listdir(WEIGHTS_DIR)):
        path = os.path.join(WEIGHTS_DIR, f)
        print(f"  {f}  ({os.path.getsize(path)} bytes)")

    # Patch hls4ml and run comparison
    patch_hls4ml()
    run_hls4ml_comparison(model, x_test, y_test, test_acc)

2026-04-14 22:43:58.017305: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-14 22:43:58.018456: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-14 22:43:58.021927: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-14 22:43:58.029188: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-14 22:43:58.039445: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been 


CNN Model Training and Weight Export
TensorFlow version: 2.17.0
Weights directory : /home/mas47153/weights

Loading MNIST dataset
  Train: (60000, 28, 28, 1)  Test: (10000, 28, 28, 1)

Building model


Model: "cnn_mnist"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1 (Conv2D)                  │ (None, 26, 26, 8)      │            80 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool1 (MaxPooling2D)            │ (None, 13, 13, 8)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2 (Conv2D)                  │ (None, 11, 11, 16)     │         1,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool2 (MaxPooling2D)            │ (None, 5, 5, 16)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 400)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc1 (Dense)                     │ (None, 64)             │        25,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc2 (Dense)                     │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 27,562 (107.66 KB)

 Trainable params: 27,562 (107.66 KB)

 Non-trainable params: 0 (0.00 B)


Training for 10 epochs
Epoch 1/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.7374 - loss: 0.9074 - val_accuracy: 0.9725 - val_loss: 0.1047
Epoch 2/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9664 - loss: 0.1151 - val_accuracy: 0.9793 - val_loss: 0.0698
Epoch 3/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9766 - loss: 0.0764 - val_accuracy: 0.9832 - val_loss: 0.0613
Epoch 4/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9808 - loss: 0.0620 - val_accuracy: 0.9847 - val_loss: 0.0516
Epoch 5/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9828 - loss: 0.0543 - val_accuracy: 0.9832 - val_loss: 0.0575
Epoch 6/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9852 - loss: 0.0467 - val_accuracy: 0.9883 - val_loss: 0.0476
Epoch 7/10
422/422 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9877 - loss: 0.0390 - val_accuracy: 0.9870 - val_loss: 0.0486
Epoch 8/10
302/422 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9895 - loss: 0